<a href="https://colab.research.google.com/github/abidahnisaa/Project_Breast_Cancer_Poster_Presentation_UNS/blob/main/2.%20Pan_Target_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# PAN-ESSENTIAL GENE IDENTIFICATION
# DepMap CRISPR Dependency Dataset
# ============================================================

import pandas as pd

# 1. Load full DepMap CRISPR dataset
df = pd.read_csv(
    "/content/drive/MyDrive/Breast Cancer Model/depmap_crispr_26Q1.csv"
)

print("Full dataset shape:", df.shape)
display(df.head())

Full dataset shape: (1208, 18537)


,model_id,cell_line_name,stripped_cell_line_name,oncotree_lineage,oncotree_primary_disease,oncotree_subtype,A1BG,AASS,ABCA10,ABCA7,...,ZNF432,ZNF516,ZNF518A,ZNF536,ZNF592,ZNF623,ZNF646,ZSCAN12,CCL4L2,CDK11B
0,ACH-001270,127399,127399,Soft Tissue,Synovial Sarcoma,Synovial Sarcoma,-0.038756,-0.215523,0.044912,0.208822,...,0.017046,NaN,NaN,0.014567,0.161617,0.019846,-0.264657,NaN,NaN,NaN
1,ACH-002680,170MGBA,170MGBA,CNS/Brain,Adult-Type Diffuse Glioma,"Glioblastoma, IDH-Wildtype",-0.007730,0.041877,0.066959,0.034664,...,0.105222,NaN,NaN,-0.022810,0.004996,-0.052879,-0.051166,NaN,NaN,NaN
2,ACH-002400,21MT1,21MT1,Breast,Invasive Breast Carcinoma,Breast Invasive Ductal Carcinoma,0.099750,0.095358,0.224868,0.108875,...,0.031457,NaN,NaN,-0.131265,0.080008,0.041570,-0.036201,NaN,NaN,NaN
3,ACH-002401,21MT2,21MT2,Breast,Invasive Breast Carcinoma,Breast Invasive Ductal Carcinoma,0.032844,0.006765,0.289841,-0.029654,...,0.021381,NaN,NaN,-0.185839,0.114350,0.047070,-0.188832,NaN,NaN,NaN
4,ACH-002399,21NT,21NT,Breast,Invasive Breast Carcinoma,Breast Invasive Ductal Carcinoma,0.095898,0.224174,0.137069,0.111254,...,0.017857,NaN,NaN,0.000366,0.198231,-0.119213,-0.073809,NaN,NaN,NaN


In [ ]:
# 2. Define metadata columns
meta_cols = [
    "model_id",
    "cell_line_name",
    "stripped_cell_line_name",
    "oncotree_lineage",
    "oncotree_primary_disease",
    "oncotree_subtype"
]

# Gene columns = all columns after metadata
gene_cols = df.columns[6:]

print("Number of cell lines:", df.shape[0])
print("Number of genes:", len(gene_cols))
print("First few genes:", list(gene_cols[:10]))

Number of cell lines: 1208
Number of genes: 18531
First few genes: ['A1BG', 'AASS', 'ABCA10', 'ABCA7', 'ABCA8', 'ABCA9', 'ABCB6', 'ABCC4', 'ABCC5', 'ABCC9']


In [ ]:
# 3. Extract gene dependency matrix
gene_df = df[gene_cols]

print("Gene dependency matrix shape:", gene_df.shape)
display(gene_df.iloc[:5, :10])

Gene dependency matrix shape: (1208, 18531)


,A1BG,AASS,ABCA10,ABCA7,ABCA8,ABCA9,ABCB6,ABCC4,ABCC5,ABCC9
0,-0.038756,-0.215523,0.044912,0.208822,-0.178665,0.084084,-0.041007,-0.241759,0.132691,0.262131
1,-0.007730,0.041877,0.066959,0.034664,0.040118,0.027607,0.018961,0.092320,0.096612,-0.020893
2,0.099750,0.095358,0.224868,0.108875,-0.098799,0.143940,0.201462,0.105788,0.222825,-0.069767
3,0.032844,0.006765,0.289841,-0.029654,-0.365346,0.072861,-0.055931,0.144662,0.113646,-0.133797
4,0.095898,0.224174,0.137069,0.111254,-0.139431,0.278303,0.099647,0.203649,0.099669,0.017622


In [ ]:
# 4. Define essentiality threshold
essential_threshold < -0.5

# TRUE = essential, FALSE = non-essential
essential_matrix = gene_df < essential_threshold

display(essential_matrix.iloc[:5, :10])

,A1BG,AASS,ABCA10,ABCA7,ABCA8,ABCA9,ABCB6,ABCC4,ABCC5,ABCC9
0,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False


In [ ]:
# 5. Calculate essentiality fraction for each gene
# TRUE = 1, FALSE = 0, so mean = fraction of essential cell lines
essential_fraction = essential_matrix.mean(axis=0)

essential_fraction_df = essential_fraction.reset_index()
essential_fraction_df.columns = ["Gene", "Essential_Fraction"]

essential_fraction_df = essential_fraction_df.sort_values(
    "Essential_Fraction",
    ascending=False
)

display(essential_fraction_df.head(20))

,Gene,Essential_Fraction
14102,CCT3,1.0
4056,ANKLE2,1.0
12527,PSMD3,1.0
12503,PSMA6,1.0
12500,PSMA3,1.0
4361,FAU,1.0
12504,PSMA7,1.0
10916,TRMT112,1.0
14980,ZNHIT2,1.0
12507,PSMB2,1.0


In [ ]:
# 6. Identify pan-essential genes
# Definition: essential in >90% of all cancer cell lines

pan_threshold = 0.9

pan_essential_df = essential_fraction_df[
    essential_fraction_df["Essential_Fraction"] > pan_threshold
].copy()

print("Number of pan-essential genes:", len(pan_essential_df))
display(pan_essential_df.head(20))

Number of pan-essential genes: 974


,Gene,Essential_Fraction
14102,CCT3,1.0
4056,ANKLE2,1.0
12527,PSMD3,1.0
12503,PSMA6,1.0
12500,PSMA3,1.0
4361,FAU,1.0
12504,PSMA7,1.0
10916,TRMT112,1.0
14980,ZNHIT2,1.0
12507,PSMB2,1.0


In [ ]:
# 7. Save pan-essential genes to Google Drive
pan_essential_df.to_csv(
    "/content/drive/MyDrive/Breast Cancer Model/pan_essential_genes.csv",
    index=False
)

print("Saved: pan_essential_genes.csv")

Saved: pan_essential_genes.csv


In [ ]:
essential_fraction_df.to_csv(
    "/content/drive/MyDrive/Breast Cancer Model/all_genes_essential_fraction.csv",
    index=False
)

print("Saved: all_genes_essential_fraction.csv")

Saved: all_genes_essential_fraction.csv
